# Record timing, time-matched landmark designs and all-feature conformal weighting

Describes record timing in the two hospital systems, evaluates the timing probe under several landmark designs and fits a domain classifier on all 242 features for weighted class-conditional conformal prediction.

Data: PhysioNet/Computing in Cardiology Challenge 2019 training sets A and B (open access, CC BY 4.0), downloaded by the first code cell.

Run on Google Colab with a CPU runtime (Runtime > Run all). All results are written to `results_01.json`, which is downloaded at the end.

## Download the data

In [ ]:
import os, glob
!pip -q install awscli
!aws s3 sync --no-sign-request --only-show-errors s3://physionet-open/challenge-2019/1.0.0/training/ /content/data/
nA = len(glob.glob('/content/data/training_setA/*.psv')); nB = len(glob.glob('/content/data/training_setB/*.psv'))
print('Set A files:', nA, ' Set B files:', nB, ' (expected 20336 and 20000)')
if nA < 20336 or nB < 20000:
    print('S3 download incomplete - using wget instead (slower)')
    !wget -r -N -c -np -q https://physionet.org/files/challenge-2019/1.0.0/training/ -P /content/
    print(len(glob.glob('/content/physionet.org/files/challenge-2019/1.0.0/training/training_setA/*.psv')),
          len(glob.glob('/content/physionet.org/files/challenge-2019/1.0.0/training/training_setB/*.psv')))

## Load all ICU stays

In [ ]:
import os, glob, json, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
DATA = None
for cand in ['/content/data', '/content/physionet.org/files/challenge-2019/1.0.0/training']:
    if cand and glob.glob(f'{cand}/training_setA/*.psv'):
        DATA = cand; break
assert DATA, 'PhysioNet files not found - run the download cell first'
print('Using data in', DATA)
DYN = ['HR','O2Sat','Temp','SBP','MAP','DBP','Resp','EtCO2','BaseExcess','HCO3','FiO2','pH','PaCO2','SaO2',
       'AST','BUN','Alkalinephos','Calcium','Chloride','Creatinine','Bilirubin_direct','Glucose','Lactate',
       'Magnesium','Phosphate','Potassium','Bilirubin_total','TroponinI','Hct','Hgb','PTT','WBC','Fibrinogen','Platelets']
H = 6          # prediction horizon (hours)
W = 6          # feature window (hours)
W_MIN = 6      # minimum observed hours before a landmark
K_CTRL = 10    # controls per septic stay in risk-set sampling
SEEDS = [2026, 2027, 2028, 2029, 2030]
OUT = {}

def load_set(s):
    files = sorted(glob.glob(f'{DATA}/training_set{s}/*.psv'))
    stays = {}
    t0 = time.time()
    for f in files:
        d = pd.read_csv(f, sep='|')
        pid = os.path.basename(f)[:-4]
        lab = d['SepsisLabel'].to_numpy()
        pos = np.flatnonzero(lab == 1)
        stays[pid] = dict(X=d[DYN].to_numpy(np.float32), n=len(d),
                          first=(int(pos[0]) if len(pos) else -1),
                          age=float(d['Age'].iloc[0]), sex=float(d['Gender'].iloc[0]),
                          hadm=float(d['HospAdmTime'].iloc[0]), icu0=float(d['ICULOS'].iloc[0]),
                          unit=(0 if d['Unit1'].iloc[0] == 1 else (1 if d['Unit2'].iloc[0] == 1 else 2)))
    print(f'Set {s}: {len(stays)} stays loaded in {time.time()-t0:.0f}s')
    return stays

S = {s: load_set(s) for s in ['A', 'B']}
assert len(S['A']) > 0 and len(S['B']) > 0, 'Data not found'

## Record-timing diagnostics

In [ ]:
diag = {}
for s, st in S.items():
    n = np.array([v['n'] for v in st.values()]); f = np.array([v['first'] for v in st.values()])
    icu0 = np.array([v['icu0'] for v in st.values()])
    sep = f >= 0
    onset = f[sep] + 6
    diag[s] = dict(stays=int(len(n)), septic=int(sep.sum()),
                   len_median_septic=float(np.median(n[sep])), len_median_nonseptic=float(np.median(n[~sep])),
                   hours_after_onset_median_septic=float(np.median(n[sep] - 1 - onset)),
                   frac_septic_record_ends_before_onset=float(np.mean(n[sep] - 1 < onset)),
                   frac_ICULOS_not_starting_at_1=float(np.mean(icu0 != 1)))
OUT['diagnostics'] = diag
print(json.dumps(diag, indent=1))

## Build cohorts under four landmark designs

In [ ]:
def septic_cases(st, h):
    rows = []
    for pid, v in st.items():
        if v['first'] < 0: continue
        tp = v['first'] + 6 - h
        if tp >= W_MIN - 1 and tp <= v['n'] - 1:
            rows.append((pid, 1, tp))
    return rows

def design_landmarks(st, h, design, seed):
    rng = np.random.default_rng(seed)
    cases = septic_cases(st, h)
    ref = np.array([c[2] for c in cases])
    negs = [(pid, v['n']) for pid, v in st.items() if v['first'] < 0]
    rows = list(cases)
    if design == 'random':            # uniform landmark for non-septic stays
        for pid, n in negs:
            if n >= W_MIN: rows.append((pid, 0, int(rng.integers(W_MIN - 1, n))))
    elif design in ('trunc', 'trunc_fu'):   # trunc = time-matched; trunc_fu = time-matched with follow-up through t + h
        for pid, n in negs:
            lim = n - 1 - (h if design == 'trunc_fu' else 0)
            cand = ref[ref <= lim]
            if len(cand): rows.append((pid, 0, int(rng.choice(cand))))
    elif design == 'riskset':         # nested case-control: controls event-free and observed through t+h
        n_arr = np.array([n for _, n in negs]); used = np.zeros(len(negs), bool)
        order = np.argsort(-ref, kind='stable'); got = []
        for i in order:
            t = ref[i]
            pool = np.flatnonzero((~used) & (n_arr - 1 >= t + h))
            take = rng.choice(pool, size=min(K_CTRL, len(pool)), replace=False) if len(pool) else []
            used[take] = True; got.append(len(take))
            rows += [(negs[j][0], 0, int(t)) for j in take]
        OUT.setdefault('riskset_full_k_fraction', {})[str(seed)] = float(np.mean(np.array(got) == K_CTRL))
    else:
        raise ValueError(design)
    return pd.DataFrame(rows, columns=['pid', 'y', 'tp'])

def features(st, lm):
    F = np.full((len(lm), 7 * len(DYN) + 4), np.nan, np.float32)
    meta = np.zeros((len(lm), 4), np.float32)   # age, sex, unit, tp (not model features)
    for r, (pid, y, tp) in enumerate(lm.itertuples(index=False)):
        v = st[pid]; hist = v['X'][:tp + 1]; win = hist[max(0, tp - W + 1):]
        with np.errstate(all='ignore'):
            ok = ~np.isnan(win); cnt = ok.sum(0)
            mean = np.nanmean(win, 0); sd = np.nanstd(win, 0); mn = np.nanmin(win, 0); mx = np.nanmax(win, 0)
            okh = ~np.isnan(hist)
            idx = np.where(okh.any(0), hist.shape[0] - 1 - np.argmax(okh[::-1], 0), -1)
            last = np.where(idx >= 0, hist[np.maximum(idx, 0), np.arange(hist.shape[1])], np.nan)
            t = np.arange(win.shape[0], dtype=np.float32)[:, None]
            tm = np.where(ok, t, 0).sum(0) / np.maximum(cnt, 1); vm = np.where(ok, win, 0).sum(0) / np.maximum(cnt, 1)
            cov = np.where(ok, (t - tm) * (np.nan_to_num(win) - vm), 0).sum(0)
            var = np.where(ok, (t - tm) ** 2, 0).sum(0)
            slope = np.where((cnt >= 2) & (var > 0), cov / np.where(var > 0, var, 1), np.nan)
            miss = 1 - cnt / win.shape[0]
        F[r, :7 * len(DYN)] = np.concatenate([mean, sd, mn, mx, last, slope, miss])
        F[r, -4:] = [v['age'], v['sex'], v['hadm'], miss.mean()]
        meta[r] = [v['age'], v['sex'], v['unit'], tp]
    return F, meta

COHORT = {}
for design in ['random', 'trunc', 'trunc_fu', 'riskset']:
    for s in ['A', 'B']:
        t0 = time.time()
        lm = design_landmarks(S[s], H, design, seed=2019 + H + (0 if s == 'A' else 100))
        X, meta = features(S[s], lm)
        COHORT[(design, s)] = dict(X=X, y=lm.y.to_numpy(), tp=lm.tp.to_numpy(), meta=meta, pid=lm.pid.to_numpy())
        print(f'{design:9s} {s}: {len(lm):6d} stays, {int(lm.y.sum()):5d} septic ({lm.y.mean():.3f}) [{time.time()-t0:.0f}s]')

## Timing probe and discrimination under each landmark design

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

def hgb(seed):
    return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                          max_depth=6, l2_regularization=1.0, class_weight='balanced',
                                          random_state=seed)

def boot_auc_diff(y, p1, p2, B=300, seed=0):
    rng = np.random.default_rng(seed); d = []
    for _ in range(B):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() == y[i].max(): continue
        d.append(roc_auc_score(y[i], p1[i]) - roc_auc_score(y[i], p2[i]))
    return [float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))]

audit = {}
for design in ['random', 'trunc', 'trunc_fu', 'riskset']:
    A, B = COHORT[(design, 'A')], COHORT[(design, 'B')]
    r = dict(nA=int(len(A['y'])), nB=int(len(B['y'])), prevB=float(B['y'].mean()))
    r['probe_tp_alone_AUROC_A'] = float(roc_auc_score(A['y'], A['tp']))
    r['probe_tp_alone_AUROC_B'] = float(roc_auc_score(B['y'], B['tp']))
    m = hgb(2026).fit(A['X'], A['y']); p = m.predict_proba(B['X'])[:, 1]
    XA_t = np.column_stack([A['X'], A['tp']]); XB_t = np.column_stack([B['X'], B['tp']])
    mt = hgb(2026).fit(XA_t, A['y']); pt = mt.predict_proba(XB_t)[:, 1]
    r.update(AUROC=float(roc_auc_score(B['y'], p)), AUPRC=float(average_precision_score(B['y'], p)),
             AUROC_with_tp=float(roc_auc_score(B['y'], pt)), AUPRC_with_tp=float(average_precision_score(B['y'], pt)))
    r['delta_AUROC_tp_CI'] = boot_auc_diff(B['y'], pt, p)
    audit[design] = r
    print(f"{design:9s} tp-probe(B)={r['probe_tp_alone_AUROC_B']:.3f}  AUROC={r['AUROC']:.3f}  "
          f"+tp={r['AUROC_with_tp']:.3f}  AUPRC={r['AUPRC']:.3f} -> {r['AUPRC_with_tp']:.3f}")
OUT['timing_audit'] = audit

## Weighted class-conditional conformal prediction with all features, both directions, 5 splits

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

def wq_thresholds(cal_s, cal_w, test_w, alpha):
    """Weighted conformal quantile (Tibshirani et al. 2019) for each test point; inf if not attainable."""
    o = np.argsort(cal_s); s = cal_s[o]; cw = np.cumsum(cal_w[o]); tot = cw[-1]
    need = (1 - alpha) * (tot + test_w)
    k = np.searchsorted(cw, need, side='left')
    thr = np.full(len(test_w), np.inf); ok = k < len(s); thr[ok] = s[k[ok]]
    return thr

def domain_weights(Xsrc_fit, Xtgt_fit, Xeval, clip_q=0.99):
    med = np.nanmedian(np.vstack([Xsrc_fit, Xtgt_fit]), 0)
    med = np.nan_to_num(med, nan=0.0)              # columns that are entirely missing -> 0
    f = lambda Z: np.nan_to_num(np.where(np.isnan(Z), med, Z), nan=0.0, posinf=0.0, neginf=0.0)
    sc = StandardScaler().fit(f(np.vstack([Xsrc_fit, Xtgt_fit])))
    Z = sc.transform(f(np.vstack([Xsrc_fit, Xtgt_fit]))); d = np.r_[np.zeros(len(Xsrc_fit)), np.ones(len(Xtgt_fit))]
    lr = LogisticRegression(C=0.1, max_iter=2000).fit(Z, d)
    pe = lr.predict_proba(sc.transform(f(Xeval)))[:, 1]
    w = pe / (1 - pe) * len(Xsrc_fit) / len(Xtgt_fit)
    auc = roc_auc_score(d, lr.predict_proba(Z)[:, 1])
    return w, auc, lr, sc, f

def run_conformal(src, tgt, seed, alphas_neg=(0.10, 0.20, 0.30), alpha_pos=0.10):
    Xs, ys = src['X'], src['y']; Xt, yt = tgt['X'], tgt['y']
    keep = (~np.isnan(Xs)).any(0) & (~np.isnan(Xt)).any(0)   # drop features entirely missing in either hospital
    Xs, Xt = Xs[:, keep], Xt[:, keep]
    tr, cc = train_test_split(np.arange(len(ys)), test_size=0.3, stratify=ys, random_state=seed)
    m = hgb(seed).fit(Xs[tr], ys[tr])
    pc = m.predict_proba(Xs[cc])[:, 1]; pt = m.predict_proba(Xt)[:, 1]; yc = ys[cc]
    sc_cal = {1: 1 - pc[yc == 1], 0: pc[yc == 0]}          # nonconformity s(x,y) = 1 - p(y|x)
    st_test = {1: 1 - pt, 0: pt}
    res = dict(AUROC=float(roc_auc_score(yt, pt)))
    # cross-fitted domain weights: classifier fitted on one half of target, applied to the other half
    rng = np.random.default_rng(seed); half = rng.random(len(yt)) < 0.5
    w_test = np.zeros(len(yt)); w_cal = np.zeros(len(cc)); aucs = []
    for h_ in (half, ~half):
        w_eval, auc, lr, scl, f = domain_weights(Xs[tr], Xt[~h_], np.vstack([Xt[h_], Xs[cc]]))
        w_test[h_] = w_eval[:h_.sum()]; w_cal += 0.5 * w_eval[h_.sum():]; aucs.append(auc)
    cap = np.quantile(np.r_[w_cal, w_test], 0.99); w_cal = np.minimum(w_cal, cap); w_test = np.minimum(w_test, cap)
    res['domain_AUROC'] = float(np.mean(aucs))
    res['ESS_frac_septic'] = float(w_cal[yc == 1].sum() ** 2 / (w_cal[yc == 1] ** 2).sum() / (yc == 1).sum())
    res['ESS_frac_nonseptic'] = float(w_cal[yc == 0].sum() ** 2 / (w_cal[yc == 0] ** 2).sum() / (yc == 0).sum())

    def evaluate(inc1, inc0, name):
        cov1 = inc1[yt == 1].mean(); cov0 = inc0[yt == 0].mean()
        alert = inc1 & ~inc0; defer = inc1 & inc0; noalert = inc0 & ~inc1
        res[name] = dict(septic_cov=float(cov1), nonseptic_cov=float(cov0),
                         set_size=float((inc1.astype(int) + inc0).mean()),
                         alert_rate=float(alert.mean()), defer_rate=float(defer.mean()),
                         empty_rate=float((~inc1 & ~inc0).mean()),
                         alert_PPV=float(yt[alert].mean()) if alert.any() else None,
                         septic_missed_as_noalert=float(noalert[yt == 1].mean()))
    # marginal split conformal at alpha = 0.10
    allc = np.r_[sc_cal[1], sc_cal[0]]; n = len(allc)
    q = np.sort(allc)[min(n - 1, int(np.ceil((n + 1) * 0.9)) - 1)]
    evaluate(st_test[1] <= q, st_test[0] <= q, 'marginal_a10')
    # Mondrian and weighted Mondrian, symmetric and asymmetric alpha
    for a0 in alphas_neg:
        thr = {}
        for y_, a in ((1, alpha_pos), (0, a0)):
            c = np.sort(sc_cal[y_]); nn = len(c)
            k = int(np.ceil((nn + 1) * (1 - a))) - 1
            thr[y_] = c[k] if k < nn else np.inf
        evaluate(st_test[1] <= thr[1], st_test[0] <= thr[0], f'mondrian_a1_{alpha_pos}_a0_{a0}')
        thrw = {y_: wq_thresholds(sc_cal[y_], w_cal[yc == y_], w_test, a) for y_, a in ((1, alpha_pos), (0, a0))}
        evaluate(st_test[1] <= thrw[1], st_test[0] <= thrw[0], f'wmondrian_a1_{alpha_pos}_a0_{a0}')
    return res

conformal = {}
for direction, (src, tgt) in {'A_to_B': ('A', 'B'), 'B_to_A': ('B', 'A')}.items():
    runs = [run_conformal(COHORT[('riskset', src)], COHORT[('riskset', tgt)], sd) for sd in SEEDS]
    keys = [k for k in runs[0] if isinstance(runs[0][k], dict)]
    summ = {k: {m: [float(np.mean([r[k][m] for r in runs if r[k][m] is not None])) if any(r[k][m] is not None for r in runs) else None,
                    float(np.min([r[k][m] for r in runs if r[k][m] is not None])) if any(r[k][m] is not None for r in runs) else None,
                    float(np.max([r[k][m] for r in runs if r[k][m] is not None])) if any(r[k][m] is not None for r in runs) else None]
                for m in runs[0][k]} for k in keys}
    for k in ('AUROC', 'domain_AUROC', 'ESS_frac_septic', 'ESS_frac_nonseptic'):
        summ[k] = [float(np.mean([r[k] for r in runs])), float(np.min([r[k] for r in runs])), float(np.max([r[k] for r in runs]))]
    conformal[direction] = summ
    print(f"\n{direction}: AUROC {summ['AUROC'][0]:.3f}, domain AUROC {summ['domain_AUROC'][0]:.3f}, "
          f"ESS septic {summ['ESS_frac_septic'][0]:.2f}")
    print(f"{'method':32s} {'septic_cov':>11s} {'nonsep_cov':>11s} {'defer':>7s} {'alert':>7s} {'PPV':>6s}")
    for k in keys:
        v = summ[k]; ppv = v['alert_PPV'][0]
        print(f"{k:32s} {v['septic_cov'][0]:6.3f}[{v['septic_cov'][1]:.2f}-{v['septic_cov'][2]:.2f}]"
              f" {v['nonseptic_cov'][0]:11.3f} {v['defer_rate'][0]:7.3f} {v['alert_rate'][0]:7.3f} "
              f"{(ppv if ppv is not None else float('nan')):6.3f}")
OUT['conformal_all_features'] = conformal

## Save results

In [ ]:
with open('results_01.json', 'w') as fh:
    json.dump(OUT, fh, indent=1)
print('Saved results_01.json')

In [ ]:
from google.colab import files
files.download('results_01.json')